# Day 6 — 생물학·생체신호 파일 형식 읽기

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/week-1/day-06.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/week-1/day-06.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — FASTA와 FASTQ를 직접 읽어 보기

In [ ]:
# --- FASTA: 이름 줄(>) + 서열 ---
fasta_text = """>geneA 가상 서열 1
ATGGCGTTAGC
CCGTTA
>geneB 가상 서열 2
ATGAAATTTGGG
"""
sequences = {}
for line in fasta_text.splitlines():
    if line.startswith(">"):
        name = line[1:].split()[0]      # '>' 뒤 첫 단어가 ID
        sequences[name] = ""
    else:
        sequences[name] += line         # 여러 줄에 걸친 서열을 이어 붙인다
for name, seq in sequences.items():
    print(name, len(seq), "bp", seq)

# --- FASTQ: read 하나 = 4줄 (ID, 서열, +, 품질) ---
fastq_text = """@read1
ACGTTGCA
+
IIIIII#5
@read2
GGCATTAC
+
IIII5555
"""
lines = fastq_text.splitlines()
for i in range(0, len(lines), 4):
    read_id, seq, _, qual = lines[i:i + 4]
    scores = [ord(ch) - 33 for ch in qual]   # Phred+33: 문자 코드에서 33을 뺀다
    print(read_id, seq, scores, "최저 품질 =", min(scores))

### 예시 2 — TSV로 저장된 count matrix 읽기

In [ ]:
import io
import pandas as pd

# 탭으로 구분된 가상 count matrix: 행 = 유전자, 열 = 샘플
tsv_text = "gene\tS1\tS2\tS3\tS4\nGfap\t120\t135\t210\t198\nSod2\t80\t76\t91\t85\nMbp\t400\t388\t250\t262\n"
counts = pd.read_csv(io.StringIO(tsv_text), sep="\t", index_col="gene")
print(counts)
print("모양(유전자 수, 샘플 수):", counts.shape)
print("샘플별 총 count:")
print(counts.sum())

### 예시 3 — WFDB header에서 기록 길이 알아내기

실제 WFDB header와 같은 형식으로 만든 가상 예시입니다. 첫 줄은 기록 전체의 정보(기록 이름, 신호 수, 샘플링 주파수, 샘플 수)이고, 그 아래는 신호(채널)마다 한 줄씩입니다.

In [ ]:
# WFDB 헤더(.hea)의 형식을 흉내 낸 가상 예시
header_text = """demo01 3 100 6000
demo01.dat 16 1000/g 16 0 12 -532 0 ACC_X
demo01.dat 16 1000/g 16 0 -3 1210 0 ACC_Y
demo01.dat 16 1000/g 16 0 998 77 0 ACC_Z
"""
lines = header_text.splitlines()
record_name, n_signals, fs, n_samples = lines[0].split()
n_signals, fs, n_samples = int(n_signals), int(fs), int(n_samples)

print("기록 이름:", record_name)
print("신호(채널) 수:", n_signals)
print("샘플링 주파수:", fs, "Hz")
print("기록 길이:", n_samples / fs, "초")
for line in lines[1:]:
    fields = line.split()
    print("채널:", fields[-1], "| 데이터 파일:", fields[0], "| gain:", fields[2])

## 직접 해보기

3. **산출물 파일을 만듭니다.**

In [ ]:
# 내 컴퓨터용 명령입니다. Colab에서는 의미가 없어 실행하지 않도록 주석 처리했습니다.
# cd ~/Documents/neurodegeneration-bioinformatics
# touch data-formats.md
# open -e data-formats.md

7. **일지와 commit.**

In [ ]:
# 내 컴퓨터용 명령입니다. Colab에서는 의미가 없어 실행하지 않도록 주석 처리했습니다.
# git add data-formats.md notebooks/01_setup_and_metadata.ipynb learning-log.md
# git commit -m "Day 6: 데이터 형식 비교표 작성"